# CKLT Analysis: Injection and Reconstruction
This notebook demonstrates the entire pipeline for generating, injecting, and analyzing a synthetic radio astronomical signal.

The pipeline consists of three phases:
1. **Generation:** Creating a `GUPPI RAW` file using `setigen` to simulate the behavior of a hardware Polyphase Filterbank (PFB).
2. **Diagnostics:** Loading the raw complex data and verifying correct frequency allocation.
3. **CKLT Processing:** Applying the Covariance Karhunen-Loève Transform to extract and reconstruct the signal submerged in thermal noise.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
import setigen as stg

from seti_klt.KLT import KLT

# Import our new logging and I/O tools
from seti_klt.utils.io import (
    SimpleLogger, 
    LogLevel, 
    prepare_output_dir, 
    save_figure
)

np.random.seed(42)

# Initialize the logger (outputs to both console and file)
logger = SimpleLogger(level=LogLevel.INFO, filename="injected_signal_pipeline.log")
logger.section("KLT Pipeline Initialization")

/mnt/c/Users/gigig/Documents/University/SETI/codes/.venv/lib/python3.12/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


Logging to: /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Injected_Signal/outputs/logs/injected_signal_pipeline.log

=== KLT PIPELINE INITIALIZATION


## 1. Physical and Structural Parameters Configuration
We define the physical characteristics of our virtual "antenna" and the processing dimensions.
We use the newly integrated `io` module to handle paths agnostically relative to the execution directory (`get_data_path()`).

In [2]:
# Simulated hardware parameters (GBT)
SAMPLE_RATE   = 3e9
NUM_TAPS      = 8
NUM_BRANCHES  = 1024
CHAN_BW       = SAMPLE_RATE / NUM_BRANCHES   # ~2.9297 MHz
FCH1_HZ       = 6e9                          # Center frequency of channel 0
NOISE_STD     = 1.0

# Signal parameters to inject
TARGET_CHANNEL = 1
BASEBAND_OFFSET_FRAC = 0.3
F_SIGNAL = FCH1_HZ + (TARGET_CHANNEL + BASEBAND_OFFSET_FRAC) * CHAN_BW

# SNR Parameters (Calibrated for pre-FFT variance)
SNR_DB    = -45.0
SNR_LIN   = 10 ** (SNR_DB / 10.0)
LEVEL     = NOISE_STD * np.sqrt(2.0 * SNR_LIN)

# KLT Parameters
WINDOW_SIZE  = 512
NUM_SAMPLES  = 2**19      

# Dynamic Path Management via io.py
local_data_dir = prepare_output_dir("outputs/raw_data")
FILE_STEM = str(local_data_dir / f"easy_test_ch{TARGET_CHANNEL}_snr{int(SNR_DB)}")
FILE_PATH = str(f"{FILE_STEM}.0000.raw")

# Log main statistics
logger.summary_stats({
    "Target Channel": TARGET_CHANNEL,
    "Injection Freq (MHz)": round(F_SIGNAL / 1e6, 4),
    "Per-sample SNR (dB)": SNR_DB,
    "Amplitude (Level)": round(LEVEL, 6),
    "Window Size (KLT)": WINDOW_SIZE,
    "Output File": FILE_PATH
}, title="Simulation Parameters")


=== SIMULATION PARAMETERS
[11:56:48] [INFO]     Target Channel: 1
[11:56:48] [INFO]     Injection Freq (MHz): 6003.8086
[11:56:48] [INFO]     Per-sample SNR (dB): -45.0000
[11:56:48] [INFO]     Amplitude (Level): 0.0080
[11:56:48] [INFO]     Window Size (KLT): 512
[11:56:48] [INFO]     Output File: outputs/raw_data/easy_test_ch1_snr-45.0000.raw


## 2. The "Synchronous Phase" Phenomenon (Why BASEBAND_OFFSET_FRAC = 0.3?)

When we apply the **C-KLT (Covariance KLT)**, the algorithm segments the 1D input array into a 2D matrix made of submatrices `WINDOW_SIZE` samples long. The first step to calculate the covariance matrix consists of **subtracting the column mean** (`matrix - col_means`).

If we chose an integer offset relative to the window (e.g., `OFFSET = 0.25` with `WINDOW_SIZE = 512`, we would have an exact number of cycles: $0.25 \times 512 = 128$ cycles per window). In this situation, the signal's phase restarts *exactly* from the same point at the beginning of each row. 
The rows become identical copies as far as the signal is concerned. Consequently:
1. The column mean `col_means` absorbs 100% of the signal's energy (coherent integration).
2. The centering operation (`matrix - col_means`) destroys and completely erases the signal from the matrix.
3. The KLT will model and extract *only noise*.

By using a fractional non-integer offset like **`0.3`** ($0.3 \times 512 = 153.6$ cycles), we force the phase to continuously "rotate" from row to row. This way, the column mean calculation tends to zero (incoherent destruction), preserving the pure signal inside the covariance matrix before extracting the eigenvectors!

In [3]:
logger.section("GUPPI RAW Generation with Setigen")
logger.info("Starting Polyphase Filterbank simulation...")

antenna = stg.voltage.Antenna(sample_rate=SAMPLE_RATE,
                               fch1=FCH1_HZ * u.Hz,
                               ascending=True,
                               num_pols=2)

# Thermal noise
antenna.x.add_noise(v_mean=0, v_std=NOISE_STD)
antenna.y.add_noise(v_mean=0, v_std=NOISE_STD)

# Polarized CW signal (90° phase shift on Y)
antenna.x.add_constant_signal(f_start=F_SIGNAL, drift_rate=0 * u.Hz/u.s, level=LEVEL, phase=0)
antenna.y.add_constant_signal(f_start=F_SIGNAL, drift_rate=0 * u.Hz/u.s, level=LEVEL, phase=np.pi/2)

digitizer   = stg.voltage.RealQuantizer(target_fwhm=32, num_bits=8)
filterbank  = stg.voltage.PolyphaseFilterbank(num_taps=NUM_TAPS, num_branches=NUM_BRANCHES)
requantizer = stg.voltage.ComplexQuantizer(target_fwhm=32, num_bits=8)

rvb = stg.voltage.RawVoltageBackend(
    antenna,
    digitizer=digitizer, filterbank=filterbank, requantizer=requantizer,
    start_chan=0, num_chans=64,
    block_size=134217728, blocks_per_file=128, num_subblocks=32,
)

# Recording .raw file
rvb.record(output_file_stem=FILE_PATH.replace('.0000.raw', ''), 
           num_blocks=1, length_mode='num_blocks',
           header_dict={'HELLO': 'easy_test', 'SRC_NAME': 'EASY_TEST', 'TELESCOP': 'GBT'},
           verbose=False)

logger.info(f"RAW file successfully written to: {FILE_PATH}")


=== GUPPI RAW GENERATION WITH SETIGEN
[11:56:50] [INFO]     Starting Polyphase Filterbank simulation...
[11:58:12] [INFO]     RAW file successfully written to: outputs/raw_data/easy_test_ch1_snr-45.0000.raw


## 3. Channel Diagnostics and Power Scan
Before processing the data "blindly", we use the KLT's diagnostic function to ensure that the injected power and the GUPPI byte semantics (`blimpy.GuppiRaw`) are consistent with what the library expects.

In [4]:
logger.section("Data Loading and Power Scan")

klt = KLT()

# Load channel (Normalization disabled if the signal is not at DC)
klt.load_data_from_guppi(FILE_PATH,
                         FILE_STEM,
                         channel=TARGET_CHANNEL,
                         num_samples=NUM_SAMPLES,
                         polarization=0)

# Execute channel power sanity check
fig_power, _, ch = klt.channel_power_scan(
    FILE_PATH,
    expected_signal_freq_Mhz=F_SIGNAL / 1e6
)

# Save figure using utils.io 
save_figure(fig_power, filename=f"{FILE_STEM}_power_scan.png")


=== DATA LOADING AND POWER SCAN

=== LOADING GUPPI DATA
[11:58:17] [INFO]     File: 'outputs/raw_data/easy_test_ch1_snr-45.0000.raw' | channel: 1 | polarization: 0 | requested samples: 524,288.


Reading blocks: 100%|██████████| 524k/524k [00:01<00:00, 283ksamp/s]


[11:58:18] [INFO]     Loaded 524,288 samples (requested: 524,288, blocks read: 1).
[11:58:18] [INFO]     File polarizations (NPOL): 4.

=== CHANNEL POWER SCAN
[11:58:18] [INFO]     File: 'outputs/raw_data/easy_test_ch1_snr-45.0000.raw' | polarization: 0 | expected signal: 6003.8086 MHz.
[11:58:21] [INFO]     Dominant channel: 0 (centre: 6000.0000 MHz, peak power: 1.200e+12).
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Injected_Signal/outputs/figures/easy_test_ch1_snr-45_power_scan.png


PosixPath('/mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Injected_Signal/outputs/figures/easy_test_ch1_snr-45_power_scan.png')

## 4. CKLT Application and Performance Evaluation
We finally apply the eigenvalue decomposition. The three plots below will provide crucial metrics:
- **Eigenspectrum**: The quantitative gap between the dominant eigenvalue (signal) and the tail of minor eigenvalues (thermal noise variance).
- **Waterfall**: Dynamic spectro-temporal visualization of the denoising effect.
- **Integrated PSD**: The frequency "slice" to evaluate by how many dB the signal peak has been lifted from the noise floor.

In [5]:
logger.section("Spectral Analysis and KLT Reconstruction")

# Apply KLT keeping a single main eigenvector
klt.apply_cklt(window_size=WINDOW_SIZE, n_eigenvectors=1)
logger.info(f"CKLT Applied (Window Size = {WINDOW_SIZE}, Components = 1)")

# 1. Eigenspectrum
fig_eig, ax_eig = klt.plot_eigenspectrum(n_components=1)
save_figure(fig_eig, filename=f"{FILE_STEM}_eigenspectrum.png")

# 2. Dynamic Spectrum (Waterfall) - figure is grabbed internally by save_figure
fig_wf, ax_wf = klt.plot_waterfall_comparison(channel = ch)
save_figure(filename=f"{FILE_STEM}_waterfall.png")

# 3. Linear PSD (Power Spectral Density)
fig_psd, axes_psd, peak = klt.plot_psd_comparison(channel = ch)
save_figure(fig_psd, filename=f"{FILE_STEM}_psd_comparison.png")

logger.info("Processing and plotting completed.")
logger.close()


=== SPECTRAL ANALYSIS AND KLT RECONSTRUCTION

=== APPLYING C-KLT
[11:58:26] [INFO]     Window size: 512 samples | windows (K): 1024 | eigenvectors to keep: 1.
[11:58:48] [INFO]     Top eigenvalue: 3939.8275 | 2nd eigenvalue: 1097.2760 | gap ratio λ₁/λ₂: 3.59x.
[11:58:48] [INFO]     Reconstruction complete — output length: 524,288 samples.
[11:58:48] [INFO]     CKLT Applied (Window Size = 512, Components = 1)
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Injected_Signal/outputs/figures/easy_test_ch1_snr-45_eigenspectrum.png
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Injected_Signal/outputs/figures/easy_test_ch1_snr-45_waterfall.png
[11:58:50] [INFO]     PSD comparison — peak noisy: 6000.8783 MHz, peak reconstructed: 6000.8783 MHz.
Figure saved → /mnt/c/Users/gigig/Documents/University/SETI/codes/notebooks/Injected_Signal/outputs/figures/easy_test_ch1_snr-45_psd_comparison.png
[11:58:51] [INFO]     Processing and plotting co